# Lezione 7 · Deep learning per il linguaggio naturale

## NLP avanzato: dalle parole ai vettori

In questa lezione costruisci, passo dopo passo, i pezzi che permettono a una rete neurale di elaborare testo: trasformi le frasi in numeri, addestri una tabella di *embedding* e usi quella tabella dentro un classificatore di testo minimale. Chiudi la sessione con un'idea ponte verso il meccanismo di attenzione, che affronterai nella prossima lezione.

**Framework:** PyTorch. Esegui questo quaderno su Google Colab senza installare nulla di aggiuntivo.

### Obiettivi di apprendimento

- Costruisci a mano un vocabolario e converti le frasi in sequenze di indici
- Crei un livello `torch.nn.Embedding` e verifichi che parole con significato simile finiscano con vettori simili dopo l'addestramento
- Addestri un classificatore di testo minimale (embedding più un livello lineare) che distingue le frasi positive da quelle negative
- Capisci perché lo schema codificatore-decodificatore è il passo precedente al meccanismo di attenzione

## Uno: dalle parole ai numeri

Una rete neurale elabora solo numeri, quindi il primo passo è sempre la tokenizzazione: dividi il testo in unità discrete (*token*) e le associ a un vocabolario, un insieme finito di simboli con un indice numerico ciascuno.

- Aggiungi due token speciali: `<PAD>` per riempire le frasi corte fino a una lunghezza comune all'interno di un batch, e `<UNK>` per qualsiasi parola che non sia nel tuo vocabolario
- Ogni frase si trasforma in una lista di indici di lunghezza fissa
- Questa rappresentazione tramite indici non dice ancora nulla sul significato: due parole simili hanno indici tanto diversi quanto due parole senza alcuna relazione. Questo problema lo risolvi con l'*embedding*, nel blocco successivo

In [ ]:
frases_ejemplo = [
    "el gato duerme en el sofa",
    "el perro juega en el jardin",
    "la nina lee un libro",
    "el nino escribe una carta",
]

# Construyes el vocabulario: cada palabra distinta recibe un indice.
# <PAD> rellena frases cortas; <UNK> cubre palabras no vistas.
palabras = sorted(set(palabra for frase in frases_ejemplo for palabra in frase.split()))
vocabulario = ["<PAD>", "<UNK>"] + palabras
indice_de = {palabra: i for i, palabra in enumerate(vocabulario)}

print(f"Vocabulario ({len(vocabulario)} palabras): {vocabulario}")


def tokenizar(frase, longitud_max):
    """Convierte una frase en una lista de indices, rellenada con <PAD>
    hasta longitud_max."""
    indices = [indice_de.get(p, indice_de["<UNK>"]) for p in frase.split()]
    indices = indices[:longitud_max]
    indices += [indice_de["<PAD>"]] * (longitud_max - len(indices))
    return indices


longitud_max = max(len(f.split()) for f in frases_ejemplo)
for frase in frases_ejemplo:
    print(f"{frase!r:35} -> {tokenizar(frase, longitud_max)}")

## Due: embedding, una mappa dei significati

Immagina di collocare ogni parola come un punto su una mappa, invece di conservarla come un'etichetta isolata. Su quella mappa, le parole con significati simili restano vicine ("re" e "regina"), e quelle senza relazione restano lontane ("re" e "strada").

- Quella mappa è lo spazio degli *embedding*, e le coordinate di ogni parola sono il suo embedding: un vettore denso di D numeri
- In PyTorch, `torch.nn.Embedding` implementa questa mappa come una tabella di V righe (una per parola del vocabolario) per D colonne
- La tabella viene inizializzata con valori casuali e si aggiusta tramite retropropagazione durante l'addestramento: nessuno decide a mano dove va ogni parola
- Lo verifichi addestrando un embedding su un piccolo corpus in cui "re" e "regina", oppure "uomo" e "donna", compaiono negli stessi contesti, e misurando la similarità tra i loro vettori prima e dopo l'addestramento

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Corpus minusculo donde "rey"/"reina" y "hombre"/"mujer" aparecen en contextos parecidos
corpus = [
    "el rey gobierna el reino",
    "la reina gobierna el reino",
    "el rey es poderoso",
    "la reina es poderosa",
    "el hombre camina por la calle",
    "la mujer camina por la calle",
    "el hombre trabaja en la oficina",
    "la mujer trabaja en la oficina",
]

palabras_emb = sorted(set(p for frase in corpus for p in frase.split()))
idx_emb = {p: i for i, p in enumerate(palabras_emb)}
V = len(palabras_emb)

# Generas pares (palabra, palabra_de_contexto) con una ventana de una posicion,
# igual que la idea de skip-gram vista en la teoria.
VENTANA = 1
pares = []
for frase in corpus:
    tokens = frase.split()
    for i, palabra in enumerate(tokens):
        for j in range(max(0, i - VENTANA), min(len(tokens), i + VENTANA + 1)):
            if i != j:
                pares.append((idx_emb[palabra], idx_emb[tokens[j]]))

entradas = torch.tensor([p[0] for p in pares], dtype=torch.long)
contextos = torch.tensor([p[1] for p in pares], dtype=torch.long)

DIM = 6
embedding = nn.Embedding(V, DIM)
salida = nn.Linear(DIM, V)  # predice la palabra de contexto a partir del embedding
optimizador = torch.optim.Adam(list(embedding.parameters()) + list(salida.parameters()), lr=0.05)


def similitud(palabra_a, palabra_b):
    va = embedding.weight[idx_emb[palabra_a]]
    vb = embedding.weight[idx_emb[palabra_b]]
    return F.cosine_similarity(va.unsqueeze(0), vb.unsqueeze(0)).item()


print("Antes de entrenar:")
print(f"  similitud(rey, reina)     = {similitud('rey', 'reina'):.3f}")
print(f"  similitud(hombre, mujer)  = {similitud('hombre', 'mujer'):.3f}")
print(f"  similitud(rey, calle)     = {similitud('rey', 'calle'):.3f}")

for epoca in range(300):
    optimizador.zero_grad()
    vectores = embedding(entradas)
    logits = salida(vectores)
    perdida = F.cross_entropy(logits, contextos)
    perdida.backward()
    optimizador.step()

print("\nDespues de entrenar:")
print(f"  similitud(rey, reina)     = {similitud('rey', 'reina'):.3f}")
print(f"  similitud(hombre, mujer)  = {similitud('hombre', 'mujer'):.3f}")
print(f"  similitud(rey, calle)     = {similitud('rey', 'calle'):.3f}")

## Tre: un classificatore di testo minimale

Con la tokenizzazione e l'embedding già pronti, costruisci un classificatore di sentimento: riceve una frase e prevede se esprime un'opinione positiva o negativa.

- L'architettura è volutamente semplice: `nn.Embedding` per trasformare ogni parola in un vettore, una media di questi vettori per riassumere l'intera frase, e un livello `nn.Linear` che produce un unico *logit*
- La media ignora l'ordine delle parole. È una semplificazione che ti permette di addestrare il modello in poche righe, ma perde informazione: lo verificherai nel quaderno di pratica
- Addestri minimizzando `BCEWithLogitsLoss`, che combina una sigmoide con l'entropia incrociata binaria in modo numericamente stabile
- Osservi la curva della perdita per verificare che il modello converga

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt

# Frases de entrenamiento con su etiqueta: uno es positivo, cero es negativo
frases = [
    "me encanta este producto",
    "excelente servicio y rapido",
    "una experiencia maravillosa",
    "lo recomiendo totalmente",
    "el equipo fue muy amable",
    "pesimo servicio y muy lento",
    "no lo recomiendo para nada",
    "una experiencia horrible",
    "el equipo fue muy grosero",
    "producto de mala calidad",
]
etiquetas = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

palabras_clf = sorted(set(p for frase in frases for p in frase.split()))
vocab_clf = ["<PAD>", "<UNK>"] + palabras_clf
indice_clf = {p: i for i, p in enumerate(vocab_clf)}
TAMANO_VOCAB = len(vocab_clf)


def tokenizar_clf(frase, longitud_max):
    indices = [indice_clf.get(p, indice_clf["<UNK>"]) for p in frase.split()]
    indices = indices[:longitud_max]
    indices += [indice_clf["<PAD>"]] * (longitud_max - len(indices))
    return indices


LONGITUD_MAX = max(len(f.split()) for f in frases)
X = torch.tensor([tokenizar_clf(f, LONGITUD_MAX) for f in frases], dtype=torch.long)
y = torch.tensor(etiquetas, dtype=torch.float32)


class ClasificadorSentimiento(nn.Module):
    def __init__(self, tamano_vocab, dim_embedding):
        super().__init__()
        # padding_idx=0 evita que <PAD> reciba gradiente: no aporta significado
        self.embedding = nn.Embedding(tamano_vocab, dim_embedding, padding_idx=0)
        self.clasificador = nn.Linear(dim_embedding, 1)

    def forward(self, x):
        vectores = self.embedding(x)          # (lote, longitud, dim_embedding)
        promedio = vectores.mean(dim=1)        # promedio de la frase completa
        logit = self.clasificador(promedio)    # (lote, 1)
        return logit.squeeze(1)


DIM_EMBEDDING = 8
modelo = ClasificadorSentimiento(TAMANO_VOCAB, DIM_EMBEDDING)

perdida_fn = nn.BCEWithLogitsLoss()  # combina sigmoide + entropia cruzada, mas estable
optimizador = optim.Adam(modelo.parameters(), lr=0.05)

historial_perdida = []
for epoca in range(200):
    optimizador.zero_grad()
    logits = modelo(X)
    perdida = perdida_fn(logits, y)
    perdida.backward()
    optimizador.step()
    historial_perdida.append(perdida.item())
    if (epoca + 1) % 50 == 0:
        print(f"Epoca {epoca + 1}: perdida = {perdida.item():.4f}")

plt.plot(historial_perdida)
plt.xlabel("Epoca")
plt.ylabel("Perdida (BCE)")
plt.title("Curva de perdida del clasificador de sentimiento")
plt.show()

In [ ]:
def predecir(frase):
    modelo.eval()
    with torch.no_grad():
        entrada = torch.tensor([tokenizar_clf(frase, LONGITUD_MAX)], dtype=torch.long)
        probabilidad = torch.sigmoid(modelo(entrada)).item()
    etiqueta = "positivo" if probabilidad > 0.5 else "negativo"
    print(f"'{frase}' -> {etiqueta} (probabilidad = {probabilidad:.3f})")


predecir("me encanta el equipo")
predecir("una experiencia pesima")
predecir("servicio amable y rapido")

## Quattro: verso i modelli sequenza-a-sequenza

Il classificatore che hai appena addestrato produce una singola etichetta a partire da un'intera frase. Ma che cosa fai quando anche l'output è una sequenza, e di lunghezza diversa da quella dell'input, come nella traduzione automatica?

- Un modello sequenza-a-sequenza combina due reti ricorrenti: un codificatore che legge la frase di origine parola per parola e riassume tutto in un unico vettore di contesto, e un decodificatore che genera la frase di destinazione a partire da quel vettore
- Il problema è il collo di bottiglia: l'intera frase di origine, che abbia cinque parole o cinquanta, viene compressa in un vettore di dimensione fissa. Più la frase è lunga, più informazione rilevante va persa
- A questo si aggiunge la difficoltà, già nota, delle reti ricorrenti nel mettere in relazione posizioni molto distanti tra loro nella sequenza
- Questa doppia limitazione è esattamente ciò che risolve il meccanismo di attenzione: nella prossima lezione smetti di forzare l'intera frase in un unico vettore e permetti al decodificatore di "guardare indietro" e pesare ogni parola dell'input in base alla sua rilevanza a ogni passo

## Riepilogo dei punti chiave

- La tokenizzazione converte il testo in indici numerici a partire da un vocabolario finito, con token speciali per il riempimento (`<PAD>`) e per le parole sconosciute (`<UNK>`)
- Un embedding sostituisce l'indice sparso con un vettore denso di D numeri, appreso tramite retropropagazione; parole con significati simili finiscono vicine in quello spazio
- Un classificatore di testo minimale combina un embedding, un riassunto della frase (qui, una media) e un livello lineare, addestrato con entropia incrociata binaria
- Calcolare la media degli embedding è semplice ma ignora l'ordine delle parole; un livello ricorrente conserva quell'informazione
- I modelli sequenza-a-sequenza (codificatore-decodificatore) risolvono compiti in cui input e output hanno lunghezze diverse, ma soffrono di un collo di bottiglia: comprimere l'intera frase in un unico vettore di contesto

**Apri ora il quaderno di pratica, `../notebooks_practica/clase07_practica_nlp_avanzado.ipynb`, e metti in pratica ciò che hai appena visto.**